# Issue #6 — outage and constraint signals

This notebook runs the reproducible Issue #6 build. It turns the EirGrid generation outage plan, transmission outage programme, and ECP constraint study into one leakage-safe 30/60-minute feature table. Raw workbooks stay in the ignored `data/raw/eirgrid/` tree; normalized data, provenance, quality checks, and the ablation decision are committed.

## Causal rule

For each prediction issue time, the builder selects only a source snapshot already published by that time. It then evaluates whether an outage is active at the prediction target using `[start, end)`. A March ECP workbook is therefore not backfilled into January training rows.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

# Work whether Jupyter starts in the repository root or notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'scripts').exists():
    REPO_ROOT = REPO_ROOT.parent
PROCESSED = REPO_ROOT / 'data' / 'processed'
BENCHMARK = REPO_ROOT / 'benchmarks' / 'outage_constraint_signals'
REPO_ROOT.name

'GridToEv-issue-6'

## Build all artifacts

The script is the single source of truth used by both command-line and notebook users. Existing raw files are checksum-verified and reused; missing files are downloaded from the official URLs in the source catalog.

In [2]:
environment = os.environ.copy()
environment['PYTHONPATH'] = str(REPO_ROOT / 'src')
completed = subprocess.run(
    [sys.executable, str(REPO_ROOT / 'scripts' / 'build_outage_constraint_data.py')],
    cwd=REPO_ROOT,
    env=environment,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

{
  "source_files": 3,
  "outage_events": 569,
  "constraint_rows": 3482,
  "feature_rows": 2867,
  "feature_columns": 63,
  "ablation_status": "rejected_development_gate",
  "production_decision": "exclude",
  "mean_mae_improvement_fraction": -0.13880709626752447
}



## Inspect coverage and quality

Availability flags distinguish a real zero from an unknown source. The quality report must show zero duplicate keys, zero horizon misalignment, zero future-publication violations, and zero infinite numeric cells.

In [3]:
quality = json.loads((PROCESSED / 'outage_constraint_quality_report.json').read_text())
pd.Series({
    'feature_rows': quality['rows'],
    'feature_columns': quality['columns'],
    'outage_events': quality['outage_event_rows'],
    'ecp_rows': quality['ecp_constraint_rows'],
    'generation_coverage': quality['generation_source_coverage'],
    'transmission_coverage': quality['transmission_source_coverage'],
    'ecp_coverage': quality['ecp_source_coverage'],
    'future_publication_violations': quality['future_publication_violations'],
})

feature_rows                     2867.000000
feature_columns                    63.000000
outage_events                     569.000000
ecp_rows                         3482.000000
generation_coverage                 1.000000
transmission_coverage               0.275201
ecp_coverage                        0.000000
future_publication_violations       0.000000
dtype: float64

In [4]:
features = pd.read_csv(PROCESSED / 'outage_constraint_features_30_60.csv')
features[[
    'issue_timestamp_utc',
    'forecast_horizon_minutes',
    'generation_total_unavailable_mw',
    'transmission_active_outage_count',
    'ecp_constraint_pressure_weighted_ratio',
]].tail()

,issue_timestamp_utc,forecast_horizon_minutes,generation_total_unavailable_mw,transmission_active_outage_count,ecp_constraint_pressure_weighted_ratio
2862,2026-01-31T21:30:00Z,30,23.0,26.0,NaN
2863,2026-01-31T21:30:00Z,60,23.0,26.0,NaN
2864,2026-01-31T22:00:00Z,30,23.0,26.0,NaN
2865,2026-01-31T22:00:00Z,60,23.0,26.0,NaN
2866,2026-01-31T22:30:00Z,30,23.0,26.0,NaN


## Development-only ablation

The final 15% test period is deliberately left sealed. The feature family enters production only if it improves mean MAE by at least 15% across development folds and does not degrade any fold. A rejected result remains useful: it prevents a weak feature group from silently reaching the deployed model.

In [5]:
ablation = json.loads((BENCHMARK / 'ablation_report.json').read_text())
pd.DataFrame(ablation['metrics']['folds']).set_index('fold')

,fit_rows,score_rows,baseline_mae_mwh,candidate_mae_mwh,mae_improvement_fraction
fold,,,,,
train_rolling_1,802,400,23.422916,23.045924,0.016095
train_rolling_2,1202,402,2.200877,2.235998,-0.015957
train_rolling_3,1604,402,9.559335,15.130165,-0.582763
validation,2006,430,15.004486,14.593403,0.027397


In [6]:
pd.Series({
    'status': ablation['status'],
    'production_decision': ablation['production_decision'],
    'mean_mae_improvement_fraction': ablation['metrics']['mean_mae_improvement_fraction'],
    'final_test_accessed': ablation['final_test_accessed'],
    'reason': ablation['reason'],
})

status                                                   rejected_development_gate
production_decision                                                        exclude
mean_mae_improvement_fraction                                            -0.138807
final_test_accessed                                                          False
reason                           Candidate did not clear the 15% mean-MAE gate ...
dtype: object